# Autoencoder — Starter Notebook
**Tarun's entry point for the ML phase of the HROC project.**

This is based on the architecture recommended by Chadwick Boulay (CB Neurotech, former Wolpaw Lab PhD):

> "Train the model in two phases. Phase 1: autoencoder (reconstruct ECoG signal). Phase 2: attach an MLP decoder that predicts H-reflex amplitude from the latent space."

## What you're building

```
Phase 1 — Autoencoder:
  ECoG signal (150 samples) → [Encoder MLP] → latent vector (32-64 dim) → [Decoder MLP] → reconstructed ECoG
  Loss: reconstruction error (MSE)

Phase 2 — H-reflex decoder:
  ECoG signal → [frozen Encoder] → latent vector → [new MLP head] → H-reflex amplitude
  Loss: MSE vs actual H-reflex measurement
```

## Why this is scientifically interesting
If the latent space shifts over the conditioning timeline (Baseline → Down-cond → Up-cond → etc.), that means the **brain is changing** as the spinal cord is conditioned. That's the novel result nobody has published.


## Step 1 — Load the data

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from decoders.decoder_2006 import decode_frag1

# Load Animal 9 — ECoG channel (ch2, index 1)
# Replace with your actual path
MYD_PATH = '../data/raw/ani-emg-eeg-9/channel_data.MYD'

signals = decode_frag1(MYD_PATH)  # (n_trials, 2, 150)
ecog = signals[:, 1, :]           # ch2 = ECoG, shape (n_trials, 150)
emg  = signals[:, 0, :]           # ch1 = SOLR EMG, shape (n_trials, 150)

print(f'ECoG shape: {ecog.shape}')
print(f'ECoG mean: {ecog.mean():.2f} µV, std: {ecog.std():.2f} µV')

## Step 2 — Normalize

In [ ]:
# Z-score normalize per trial
mean = ecog.mean(axis=1, keepdims=True)
std  = ecog.std(axis=1, keepdims=True) + 1e-8
ecog_norm = (ecog - mean) / std

# Train/val split
n = len(ecog_norm)
n_train = int(0.85 * n)
X_train = torch.tensor(ecog_norm[:n_train], dtype=torch.float32)
X_val   = torch.tensor(ecog_norm[n_train:], dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train), batch_size=256, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val),   batch_size=256)

print(f'Train: {len(X_train)} trials | Val: {len(X_val)} trials')

## Step 3 — Define the autoencoder

Chad said: **multi-layer perceptron encoder, encode the latent signal.**
Keep it simple first — 3 layers in, 3 layers out.

In [ ]:
class HROCAutoencoder(nn.Module):
    def __init__(self, input_dim=150, latent_dim=32):
        super().__init__()

        # Encoder: 150 → 128 → 64 → latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim),
        )

        # Decoder: latent_dim → 64 → 128 → 150
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim),
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z


model = HROCAutoencoder(input_dim=150, latent_dim=32)
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## Step 4 — Train Phase 1 (autoencoder)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

train_losses, val_losses = [], []
EPOCHS = 50

for epoch in range(EPOCHS):
    # Train
    model.train()
    batch_losses = []
    for (x,) in train_loader:
        x = x.to(device)
        recon, z = model(x)
        loss = criterion(recon, x)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())
    train_losses.append(np.mean(batch_losses))

    # Val
    model.eval()
    with torch.no_grad():
        val_loss = np.mean([criterion(model(x.to(device))[0], x.to(device)).item() for (x,) in val_loader])
    val_losses.append(val_loss)

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} | Train: {train_losses[-1]:.4f} | Val: {val_loss:.4f}')

# Plot loss
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Val')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Autoencoder Training')
plt.legend()
plt.tight_layout()
plt.savefig('autoencoder_loss.png', dpi=150)

## Step 5 — Visualize latent space (UMAP)

This is the key scientific check: do trials from different conditioning phases cluster separately in the latent space?
If yes — the brain signal carries conditioning information.

In [ ]:
import umap

model.eval()
with torch.no_grad():
    X_all = torch.tensor(ecog_norm, dtype=torch.float32).to(device)
    latents = model.encode(X_all).cpu().numpy()  # (n_trials, 32)

print(f'Latent space shape: {latents.shape}')

# TODO: add phase labels once recovered from Animal 9 log annotations
# phase_labels = ...  # array of int, one per trial
# phase_names = ['Baseline', 'Down-cond 1', 'Up-cond 1', 'Freely Running', 'Down-cond 2', 'Up-cond 2']

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
embedding = reducer.fit_transform(latents)  # (n_trials, 2)

plt.figure(figsize=(8, 6))
# Color by trial index as proxy for time until phase labels are added
sc = plt.scatter(embedding[:, 0], embedding[:, 1],
                 c=np.arange(len(embedding)), cmap='viridis',
                 s=1, alpha=0.5)
plt.colorbar(sc, label='Trial index (proxy for time)')
plt.title('Latent space — ECoG trials (Animal 9)\nColored by time — replace with phase labels')
plt.tight_layout()
plt.savefig('latent_umap.png', dpi=150)
print('Saved latent_umap.png')

## Next steps for Tarun

1. **Fix baseline bug** in `decoder_2006.py` — read frag0 from preceding record
2. **Recover phase labels** from Animal 9 log annotations (same as Animal 16/17 type-15 records)
3. **Color the UMAP** by conditioning phase — does it separate?
4. **Add more animals** — ani-emg-eeg-10, -10i for more training data
5. **Phase 2** — attach MLP head and predict H-reflex amplitude from latent space
6. **Convert data to Zarr** using the SQLAlchemy loader for faster cloud training

Ask Suchith before changing any decoder parameters — the binary formats are fragile.